## 1. torch_prep_kfold.py --initial_split  → writes *_trn_final.csv, *_tst_preprocess.csv

Typical Workflow


Step 1: Initial split (from raw features + labels)
python torch_prep_kfold.py \
  --initial_split \
  --model_type reg \
  --prefix gbsa \
  --ref_file ref.csv \
  --ref_id_col sequence \
  --ref_label_col bind_avg \
  --filenames features1.csv features2.csv \
  --feature_id_col sequence \
  --test_percentage 0.15 \
  --scramble_fractions 0.0


In [1]:
import sys
import logging
import argparse
import numpy as np
import pandas as pd
from typing import Any, Optional, List, Tuple
from sklearn.model_selection import GroupKFold, StratifiedGroupKFold
import os

In [17]:
random_state = 42
test_percentage = 0.15

In [7]:
###############################################################################
# Data Loading (Reference + Features)
###############################################################################

In [ ]:
id_col = 'sequence'
label_col = 'bind_avg'
df1 = pd.read_csv('exp_data_all.csv')
ref_data = df1[[id_col, label_col]].copy()

print(ref_data)

                                 sequence  bind_avg
0    GTACACAATTTTTTACAAAATTTAAATTAAAACAAA -0.862667
1    GCAGCCGAGGCGGAGAGAGAGAGAGGACAGCTTACG -0.703319
2    AGGCCCAGGAAGAACAATGGCTCTGCCAACTGGGCA -0.659464
3    CTTCCTCACCTGCAGACTTCCTTCCCTGAGTCCCAG -0.543823
4    TGAGGGTCAGAGGCACCCCTTCCTGGAATCTCCTTC -0.424828
..                                    ...       ...
163  ATCTCCTGGGGCGACCACGAGGTCACCCGTCCAGGT  1.524243
164  GAAAACCAGCGAGACCGCATGGTCTCACTTATAAGT  1.452711
165  AGGGAGTTCTCACACCATGTGGGTGGGATTGTAACT  1.305160
166  ACACTGAGCTTCCTCCACGTGCCCAGGTCCTGGCAG  1.430431
167  GTGTCTCCATTGGGGCACGTGTTTATATGTTTATAA  1.598717

[168 rows x 2 columns]


In [10]:
usecols = ['sequence','run','VDWAALS','EEL','EGB','ESURF','HB Energy','Hydrophobic Energy','Pi-Pi Energy','Delta_Entropy']

df2 = pd.read_csv('rawdat.csv', usecols=usecols)
feature_data = df2.copy()

print(feature_data)

                                    sequence  run  VDWAALS       EEL  \
0       GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC    9 -252.110 -1886.830   
1       GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC    9 -238.510 -1881.424   
2       GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC    9 -246.721 -1895.687   
3       GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC    9 -235.671 -1857.573   
4       GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC    9 -230.214 -1897.268   
...                                      ...  ...      ...       ...   
272155  TTTTTTTTTTTTTTGAGAAAATGAAGACAATTATCT   20 -148.291 -1941.978   
272156  TTTTTTTTTTTTTTGAGAAAATGAAGACAATTATCT   20 -142.375 -1959.709   
272157  TTTTTTTTTTTTTTGAGAAAATGAAGACAATTATCT   20 -167.433 -1911.650   
272158  TTTTTTTTTTTTTTGAGAAAATGAAGACAATTATCT   20 -136.663 -1920.439   
272159  TTTTTTTTTTTTTTGAGAAAATGAAGACAATTATCT   20 -146.999 -1929.330   

             EGB   ESURF  HB Energy  Hydrophobic Energy  Pi-Pi Energy  \
0       1841.253 -36.482  -1.940432         -165.447020 -1.655

In [8]:
## I am not doing sequence level scrambling

1) --initial_split:
   - Merges feature data (optionally from multiple files) with reference data using a shared ID column.
   - Performs an optional "sequence-level scrambling" of labels in the TRAINING set only, controlled by `scramble_fractions`.
   - Splits into train and test sets by unique sequence ID (or stratified for classification).
   - Saves the resulting CSV files:
       * PREFIX_MODELTYPE_scrFRAC_trn_final.csv   (training)
       * PREFIXMODELTYPE_scrFRAC_tst_preprocess.csv (test)

In [13]:
df_merged = pd.merge(feature_data, ref_data, on=id_col, how="inner")
print(df_merged.head())

                               sequence  run  VDWAALS       EEL       EGB  \
0  CAGGGCTGGGTCCACCTCATGGCCTTTGTTCTGGAA    9 -236.997 -1869.660  1823.216   
1  CAGGGCTGGGTCCACCTCATGGCCTTTGTTCTGGAA    9 -218.620 -1850.331  1807.831   
2  CAGGGCTGGGTCCACCTCATGGCCTTTGTTCTGGAA    9 -232.611 -1878.075  1834.181   
3  CAGGGCTGGGTCCACCTCATGGCCTTTGTTCTGGAA    9 -203.677 -1870.595  1823.641   
4  CAGGGCTGGGTCCACCTCATGGCCTTTGTTCTGGAA    9 -212.279 -1864.730  1820.462   

    ESURF  HB Energy  Hydrophobic Energy  Pi-Pi Energy  Delta_Entropy  \
0 -35.292  -2.590101         -156.445725     -4.282747     -24.750849   
1 -32.521  -2.977171         -142.709472     -7.240534     -25.235404   
2 -34.170  -3.105868         -145.088977     -8.856276     -25.124940   
3 -32.402  -3.414769         -150.961716     -5.338670     -23.079573   
4 -31.858  -3.571942         -146.583284     -7.171679     -22.812241   

   bind_avg  
0  0.166339  
1  0.166339  
2  0.166339  
3  0.166339  
4  0.166339  


In [19]:
# Shuffle everything
df_merged = df_merged.sample(frac=1.0, random_state=random_state).reset_index(drop=True)
print(df_merged.head(), df_merged.shape)

                               sequence  run  VDWAALS       EEL       EGB  \
0  CCGAGGAGGGCGGACCACGAGGTCAGGAGATGGAGA    2 -218.416 -1894.293  1848.903   
1  CAAAGGATCATGAAGATTGAGGTTTCCAGACCTTGC    4 -209.253 -1906.278  1859.001   
2  CCGAGGAGGGCGGACCACGAGGTCAGGAGATGGAGA   14 -227.307 -1915.322  1866.753   
3  TGTGGAGCAAGGGAGACAGAAGCTCATTGGCTAGAG   20 -196.044 -1881.560  1835.269   
4  AGGTAGTTTTCATAGTTTTTTTTTTTTTAACTTTTT   12 -157.573 -1845.095  1802.986   

    ESURF  HB Energy  Hydrophobic Energy  Pi-Pi Energy  Delta_Entropy  \
0 -31.807  -2.657492         -140.896396     -0.077802     -21.417399   
1 -29.121 -11.019891         -127.966819     -1.289477     -21.351275   
2 -33.252  -2.612685         -150.511490     -1.203087     -25.476614   
3 -30.107 -13.739026         -132.036809     -3.231077     -21.621044   
4 -24.038  -1.764908          -96.266936     -1.988610     -18.792626   

   bind_avg  
0  1.529260  
1  0.092349  
2  1.529260  
3 -0.722719  
4 -0.280619   (68040, 11)


In [18]:
# Split train vs test by sequence or stratified group
## for regression, not bin or mclass

from sklearn.model_selection import GroupKFold

unique_seqs = df_merged[id_col].unique() #unique sequences are extracted
np.random.seed(random_state)
np.random.shuffle(unique_seqs) #randomly shuffles unique sequences

n_train = int((1 - test_percentage) * len(unique_seqs)) # Compute train/test split boundary

train_seqs = unique_seqs[:n_train] # First 85% (after shuffle) = training sequence IDs.
test_seqs  = unique_seqs[n_train:]

# Filter the rows accordingly
df_train = df_merged[df_merged[id_col].isin(train_seqs)].copy()
df_test  = df_merged[df_merged[id_col].isin(test_seqs)].copy()

print(df_train.shape, df_test.shape)


(56700, 11) (11340, 11)


In [20]:
if "run" in df_train.columns:
    df_train.drop(columns=["run"], inplace=True, errors="ignore")
if "run" in df_test.columns:
    df_test.drop(columns=["run"], inplace=True, errors="ignore")

print(df_train.shape, df_test.shape)

(56700, 10) (11340, 10)


In [ ]:
# Save
train_file = f"reg_scr{frac_str}_trn_final.csv"
test_file  = f"reg_scr{frac_str}_tst_preprocess.csv"

df_train.to_csv(train_file, index=False)
df_test.to_csv(test_file, index=False)

logging.info(f"Scr={scr_frac}: train => {train_file}, shape={df_train.shape}")
logging.info(f"Scr={scr_frac}: test  => {test_file}, shape={df_test.shape}")
logging.info("Initial split completed.")
